# SPOD to DQM Excel
- Prerequisites: 
  - anaconda packages: `xlsxwriter, openpyxl, pandas`

## Result

Excel sheet containing:



## Configuration
The following parameters has to be definded when running as regular python script

In [ ]:
LIBRARY = '../../pythonWork/pythonSource'
DESTINATION = 'DQM_mapping.xlsx'
MODEL_SOURCE = '/Users/bue/projects/geberit/DEAP/DB/IM_GEBERIT.json'

## Check prerequisites

In [ ]:
import sys
import logging
import os
import json
import yaml
from pathlib import Path

In [ ]:
import xlsxwriter

In [ ]:
# openpyxl
from openpyxl import Workbook
from openpyxl.worksheet.table import Table
from openpyxl.utils.cell import get_column_letter
from openpyxl.styles import PatternFill

In [ ]:
spod_file = Path(MODEL_SOURCE)
assert spod_file.is_file(), f"Cannot find SPOD file '{spod_file.resolve()}'"

with open(spod_file, 'r') as src:
    spod = json.load(src)
assert spod['model'] is not None
print(f"Loaded SPOD containing {spod['model']} from '{spod_file.resolve()}'")

In [ ]:
print(f"Loaded {spod_file.resolve()}\n{spod['model']}\nVersion {spod['_imprint_']}")
print(f"Languages: {list(spod['languages'].keys())}")
mapdict = {}
for entry in ['entities', 'attributes', 'systems', 'columns']:
    print(f"- {entry}: {len(spod[entry])}")

## Initialize logging

In [ ]:
import logging
from logging import handlers
from datetime import datetime

stamp = datetime.now()
run_stamp = stamp.strftime("%Y-%m-%d-%H-%M-%S")

os.makedirs('log', exist_ok=True)
logfile = f'log/geberit-dqm-export-{run_stamp}.log'

handler = handlers.RotatingFileHandler(logfile, maxBytes=(1024 * 1024 * 10), backupCount=10)
handler.setLevel(logging.DEBUG)

formatter = logging.Formatter("%(asctime)s [%(threadName)s] - %(name)s - %(levelname)s - %(message)s")
handler.setFormatter(formatter)

console_log_handler = logging.StreamHandler()
console_formatter = logging.Formatter("%(levelname)s - %(message)s")
console_log_handler.setFormatter(console_formatter)
console_log_handler.setLevel(logging.INFO)

logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
logger.addHandler(handler)
logger.addHandler(console_log_handler)

## Use the fyayc SPOD library

In [ ]:
toolpath = Path(LIBRARY)
assert toolpath.is_dir(), f"{toolpath.reslove()} is not a directory. The constant 'LIBRARY' must point to the library. Default = 'pythonWork/pythonSource'."
sys.path.insert(0, str(toolpath))

from PUBLISH_MODEL.excel.mapping_publisher import generate
from SSOT_infra.translator import Translator

## Translation shortcut tr

In [ ]:
translator = Translator('de')

## Sheet structure definition

In [ ]:
column_headers = [
        "ID_Merkmal_Attribut",
        "Name", 
        "System", 
        "Hauptsource", 
        "Entität", 
        "Mandatory", 
        "Datenfeld_Verantwortung", 
        "Fachspezialist_für_Datenlieferungen",
        "Deadline_Merkmal_Attribut",
        "MSTAE_vorhanden_ab",
        "MSTAE_prüfbar_ab",
        "Klassifikation_Gruppierung",
        "Untergruppierung_Paketierung_DQM",
        "ID_Fremdsystem",
        "Fremdsystem",
        "Info_DQM",
        "IM-Key",
        "Mapped-Columns",
    ]

In [ ]:
def row_emitter(spod, key, main, aux, mapped) -> [str]:
    """Generate one row as defined by the headers
    @param spod the whole spod
    @param main SPOD column object
    """
    result = [
        main['interface_col_id'],
        main['name'],
        main['interface-name+'].replace('DM_', ''),
        main['interface-name+'].replace('DM_', ''),
        main['table-name+'],
        'Ja' if main['mandatory'] else 'Nein',
        None,
        None,
        None,
        None,
        None,
        None,
        None,
        aux['interface_col_id'],
        aux['interface-name+'].replace('DM_', ''),
        'DQM rocks 😀',
        key,
        ', '.join(mapped),
    ]
    return result

In [ ]:
print(f"Generating Excel with {len(column_headers)} columns [A:{get_column_letter(len(column_headers))}]")

In [ ]:
test = row_emitter(spod, 'COLU4715', spod['columns']['COLU4715'], spod['columns']['COLU4046'], [])
assert len(test) == len(column_headers)

## Row generator

In [ ]:
def system_id(spod: dict, name: str) -> str:
    hits = filter(lambda s: s[1]['name'] == name, spod['systems'].items())
    return next(hits)[0]

def aux_column(spod: dict, column: dict) -> dict:
    """
    Find secondary column with the same DQM checks on the other system (DM_STEP <-> DM_SAP)
    TODO: Align on similiarity?
    """
    own_system = column.get('interface-name+')
    other_system_id = system_id(spod, 'DM_STEP' if own_system == 'DM_SAP' else 'DM_SAP')

    mapped_attributes = column.get('attributesmapped')
    if len(mapped_attributes) > 0:
        logging.debug(f"Scannning for mapping to interface {other_system_id}")
        parent_attributes = set(mapped_attributes)
        hits = filter(lambda c: len(parent_attributes.intersection(set(c[1].get('attributesmapped', [])))) > 0, spod['columns'].items())
        other_hits = filter(lambda c: c[1]['interface-id+'] == other_system_id, hits)
        candidates = set(map(lambda c: c[0], other_hits))
        if len(candidates) > 0:
            key = next(iter(candidates))
            return spod['columns'][key], candidates
        
    return ({ 'interface_col_id': None, 'interface-name+': '' }, [])

In [ ]:
aux_column(spod, spod['columns']['COLU4715'])

In [ ]:
def row_generator() -> tuple:
    for key, column in filter(lambda t: t[1]['interface-name+'] in ['DM_SAP', 'DM_STEP'], spod['columns'].items()):
        techid = column['interface_col_id'].strip()
        if techid == '.' or len(techid) == 0:
            logging.warning(f"Hiding column {key} as it does not reference a column but a table")
            continue
        aux, mapped = aux_column(spod, column)
        logging.debug(f"Found {len(mapped)} mappings to for column {key}")
        yield (spod, key, column, aux, mapped)

# Create data table (content)

In [ ]:
data_table = [ row_emitter(*t) for t in row_generator() ]

In [ ]:
print(f"{len(data_table)} rows")

In [ ]:
### Sort by Attribute FQN
# data_table.sort(key=lambda r: r[0] if r[0] is not None else '\uFFFF')

# Write excel

In [ ]:
xlsx_destination = Path(DESTINATION)
print(f"Destination Excel sheet: '{xlsx_destination}")

## Prepare Mapping sheet

In [ ]:
workbook = xlsxwriter.Workbook(xlsx_destination)

title_format = workbook.add_format({'bold': True, 'font_color': 'black', 'font_size': 20})
column_head_format = workbook.add_format({'bold': True, 'bg_color': '#A0A0A0'})

worksheet = workbook.add_worksheet('Mapping')

col = 0
for header in column_headers:
    worksheet.write(0, col, header)
    col += 1

row = 1
for entry in data_table:
    col = 0
    for item in entry:
        worksheet.write(row, col, item)
        col += 1
    row += 1

### Define Table

In [ ]:
table_column_headers = [ { 'header': name } for name in column_headers ]

In [ ]:
worksheet.add_table(0, 0, len(data_table) + 1, len(column_headers) - 1, { 
    'name': 'mapping',
    'banded_rows': True,
    'columns': table_column_headers,
})

### Styling

In [ ]:
%%script false --no-raise-error

# FQN width
worksheet.set_column(0, 0, 20)

# Hide EID, AID on the left
worksheet.set_column(1, 3, 10, None, { 'hidden': 1, })

# EID, AID
worksheet.set_column(3, 5, 20)

# Hide attribute name translations (DE, FR)
worksheet.set_column(5, 8, 40, None, { 'hidden': 1, })

base = len(headings_im)
index = 0
for system in spod['systems'].values():
    colnr = base + (index * 3)
    worksheet.set_column(colnr, colnr, None, None, { 'hidden': 1, })
    
    # Column name on system
    worksheet.set_column(colnr + 1, colnr + 1, 30)
    
    # Technical reference
    worksheet.set_column(colnr + 2, colnr + 2, None, None, { 'hidden': 1, })        
    index += 1

## Styling

In [ ]:
%%script false --no-raise-error

summary.set_column(0, 0, 20)
summary.set_column(1, 1, 60)
summary.set_column(2, 5, 15)

## Write Excel file

In [ ]:
import SSOT_infra.parameters as parameters

In [ ]:
workbook.set_properties({
    'title':    f"{spod['model']['name']}",
    'subject':  'mapping',
    'author':   f"Excel Mapping Publisher {parameters.toolversion()}",
#    'manager':  'D',
#    'company':  'of Wolves',
    'category': 'export',
    'keywords': 'Information Model, Data Models',
    'comments': f"generated with {parameters.toolversion()} from model {spod['_imprint_']['Modelversion']}",
    'status':   'Draft',
    'revision': spod['_imprint_'].get('git-revision')
})

In [ ]:
workbook.close()
print(f"Wrote {xlsx_destination}")

# Visually verify

In [ ]:
import subprocess
r = subprocess.run(['qlmanage', '-x', '-p', xlsx_destination], shell=False) # capture_output=False, stderr=subprocess.DEVNULL)

In [ ]:
import pandas
excel_data_df = pandas.read_excel(DESTINATION, sheet_name='Mapping')

In [ ]:
from IPython.display import display, HTML
display(excel_data_df)